# Final Comparison — 3 Models
**Kết hợp kết quả từ Baseline 1, Baseline 2, và Proposed**  
Chạy notebook này SAU KHI đã train xong cả 3 model và download các file `.json` về.

**Upload các file sau vào dataset hoặc working directory:**
- `evaluation_baseline1.json`
- `evaluation_baseline2.json`
- `evaluation_proposed.json`

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

In [ ]:
# ── Load results ──
DATA_DIRS = [
    Path('/kaggle/working/models'),
    Path('/kaggle/working'),
    Path('/kaggle/input'),
]
files = [
    'evaluation_baseline1.json',
    'evaluation_baseline2.json',
    'evaluation_proposed.json',
]
results = []
for f in files:
    for d in DATA_DIRS:
        p = d / f
        if p.exists():
            with open(p) as fp:
                results.append(json.load(fp))
            print(f'Loaded: {p}')
            break

print(f'\nLoaded {len(results)} models')

In [ ]:
# ── Metrics can hien thi ──
names        = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr']
model_labels = ['Baseline 1\n(ISCX)', 'Baseline 2\n(Mendeley URL)', 'Proposed\n(Full)']
model_colors       = ['#1f77b4', '#e41a1c', '#33a02c']
model_colors_light = ['#d5e8ff', '#ffd5d5', '#d5f5d5']

# ── Print bang ket qua ──
print('\n' + '='*90)
header = f'{"Model":<35}'
for n in names:
    header += f' {n.upper():<16}'
print(header)
print('='*90)
for r, label in zip(results, model_labels):
    line = f'{r.get("model","")[:34]:<35}'
    for n in names:
        v = r.get(n, 0.0)
        s = r.get(f'{n}_std', 0.0)
        line += f' {v:.4f}+-{s:.4f}  '
    print(line)
print('='*90)

# ── Delta: Proposed vs Baseline 2 ──
print('\n=== IMPROVEMENT: Proposed vs Baseline 2 ===')
b2, prop = results[1], results[2]
for n in names:
    delta = prop.get(n, 0.0) - b2.get(n, 0.0)
    sign  = '+' if delta >= 0 else ''
    good  = 'up better' if (n != 'fpr' and delta > 0) or (n == 'fpr' and delta < 0) else 'down'
    print(f'  {n.upper():10s}: {sign}{delta:.4f}  {good}')

In [ ]:
# ══════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════
fig = plt.figure(figsize=(22, 7))

# ── 1. Grouped Bar Chart (co error bar) ──
ax0 = fig.add_subplot(1, 3, 1)
x = np.arange(len(names))
w = 0.25
for i, r in enumerate(results):
    means = [r.get(n, 0.0) for n in names]
    errs  = [r.get(f'{n}_std', 0.0) for n in names]
    ax0.bar(x + (i-1)*w, means, w, yerr=errs, capsize=3,
            label=model_labels[i], color=model_colors[i], alpha=0.85)
ax0.set_xticks(x)
ax0.set_xticklabels([n.upper() if n == 'fpr' else n.capitalize() for n in names])
ax0.set_ylim(0, 1.1)
ax0.set_ylabel('Score')
ax0.set_title('3-Model Comparison (Mean +- Std)')
ax0.legend(fontsize=8)
ax0.grid(axis='y', alpha=0.3)
ax0.axhline(y=0.5, color='gray', ls='--', alpha=0.3)

# ── 2. Radar Chart (FPR dao nguoc thanh 1-FPR) ──
radar_names  = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr']
radar_labels = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC', '1-FPR']
angles = np.linspace(0, 2*np.pi, len(radar_names), endpoint=False).tolist()
angles += angles[:1]

ax1 = fig.add_subplot(1, 3, 2, projection='polar')
for i, r in enumerate(results):
    vals = []
    for n in radar_names:
        v = r.get(n, 0.0)
        vals.append(1 - v if n == 'fpr' else v)
    vals += vals[:1]
    ax1.plot(angles, vals, 'o-', color=model_colors[i], lw=2,
             label=model_labels[i], ms=5)
    ax1.fill(angles, vals, alpha=0.08, color=model_colors[i])

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(radar_labels, fontsize=9)
ax1.set_ylim(0.8, 1.0)
ax1.set_title('Metrics Radar\n(1-FPR: cao = tot)', pad=20)
ax1.legend(fontsize=8, loc='upper right', bbox_to_anchor=(1.45, 1.15))

# ── 3. Table (mau nhat de doc) ──
ax2 = fig.add_subplot(1, 3, 3)
ax2.axis('off')
col_labels = [n.upper() if n == 'fpr' else n.capitalize() for n in names]
cell_text  = [[f'{r.get(n, 0.0):.4f}' for n in names] for r in results]
tbl = ax2.table(
    cellText=cell_text,
    rowLabels=model_labels,
    colLabels=col_labels,
    loc='center', cellLoc='center',
    rowColours=model_colors_light,
    colColours=['#e8e8e8'] * len(names)
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 2.0)
ax2.set_title('Metrics Table', fontsize=13, pad=20)

plt.suptitle('Phishing Detection - 3-Model Ablation Study',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

out = Path('/kaggle/working') / 'model_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

### Ket luan
- **Baseline 1:** TabTransformer chi tren 29 features (ISCX)
- **Baseline 2:** TabTransformer chi tren 12 URL features (Mendeley)
- **Proposed:** Gated Fusion (URL + ModernBERT text + DOM)

Download `model_comparison.png` tu Output tab.